In [1]:
import sys
from pathlib import Path

# Get the absolute path of the project root (one level above notebooks/)
project_root = Path().resolve().parent

# Add "src/" to sys.path
sys.path.append(str(project_root / "src"))

In [2]:
# Toy model (cube example)

import jax
import jax.numpy as jnp
import os
import matplotlib.pyplot as plt
from pinn.train import create_train_state, train_step
from pinn.cryoet_io import generate_synthetic_cryoET
from pinn.plot import visualize_results

# Initialize JAX random key
key = jax.random.PRNGKey(0)

# Set loss function weights
lambda_1 = 50000.0  # Weight for data loss
lambda_2 = 0.005    # Weight for physics loss

# Create train state with custom loss weights
state, model = create_train_state(key, lambda_1=lambda_1, lambda_2=lambda_2)

# Check that the loss weights are stored
print(f"Using lambda_1: {state.lambda_1}, lambda_2: {state.lambda_2}")

# Generate synthetic cryo-ET data
cryoET_data = generate_synthetic_cryoET()

# Training Data
x_train = jax.random.uniform(key, (1000, 3)) * 3 - 1.5  # Sampled from [-1.5, 1.5]^3

# Training loop
num_steps = 500
save_interval = 100

for step in range(num_steps):
    state, loss_val, loss_data_val, loss_physics_val = train_step(state, x_train, cryoET_data)

    if step % save_interval == 0:
        print(f"Step {step}, Total Loss: {loss_val:.6f}, Data Loss: {loss_data_val:.6f}, Physics Loss: {loss_physics_val:.6f}")
        plt.figure(figsize=(6,6))
        visualize_results(lambda x: state.apply_fn(state.params, x), step=step, cryoET_data=cryoET_data)
        plt.savefig(f"../outputs/figs/step_{step:04d}.png")
        plt.close()




Using lambda_1: 50000.0, lambda_2: 0.005
Step 0, Total Loss: 900.769470, Data Loss: 0.017991, Physics Loss: 244.455231
Step 100, Total Loss: 40.854038, Data Loss: 0.000674, Physics Loss: 1426.127686
Step 200, Total Loss: 21.407919, Data Loss: 0.000335, Physics Loss: 931.050720
Step 300, Total Loss: 12.682705, Data Loss: 0.000190, Physics Loss: 632.696045


KeyboardInterrupt: 

<Figure size 600x600 with 0 Axes>

<Figure size 600x600 with 0 Axes>

<Figure size 600x600 with 0 Axes>

<Figure size 600x600 with 0 Axes>

In [ ]:
import jax
import jax.numpy as jnp
import os
import matplotlib.pyplot as plt
from pinn.train import create_train_state, train_step
from pinn.cryoet_io import load_mrc_data
from pinn.plot import visualize_results

# Initialize JAX random key
key = jax.random.PRNGKey(0)

# Set loss function weights
lambda_1 = 50000.0  # Weight for data loss
lambda_2 = 0.005    # Weight for physics loss

# Create train state with custom loss weights
state, model = create_train_state(key, lambda_1=lambda_1, lambda_2=lambda_2)

# Check that the loss weights are stored
print(f"Using lambda_1: {state.lambda_1}, lambda_2: {state.lambda_2}")

# Load MRC data 
mrc_file_path = "../data/synthetic/biconcave.mrc"
cryoET_data = load_mrc_data(mrc_file_path, grid_size=64)
print("MRC data loaded successfully! Shape:", cryoET_data.shape)


print("Cryo-ET Data Shape:", cryoET_data.shape)
print("Cryo-ET Data Min:", cryoET_data.min())
print("Cryo-ET Data Max:", cryoET_data.max())
print("Cryo-ET Unique Values:", jnp.unique(cryoET_data))  # To check if it's binary (0/1) or has other values


# Training Data
x_train = jax.random.uniform(key, (1000, 3)) * 3 - 1.5  # Sampled from [-1.5, 1.5]^3

# Training loop
num_steps = 500
save_interval = 100

for step in range(num_steps):
    state, loss_val, loss_data_val, loss_physics_val = train_step(state, x_train, cryoET_data)

    if step % save_interval == 0:
        print(f"Step {step}, Total Loss: {loss_val:.6f}, Data Loss: {loss_data_val:.6f}, Physics Loss: {loss_physics_val:.6f}")
        plt.figure(figsize=(6,6))
        visualize_results(lambda x: state.apply_fn(state.params, x), step=step, cryoET_data=cryoET_data)
        plt.savefig(f"../outputs/figs/step_{step:04d}.png")
        plt.close()




Using lambda_1: 50000.0, lambda_2: 0.005
Resizing MRC data from (128, 128, 128) to (64, 64, 64)
MRC data loaded successfully! Shape: (64, 64, 64)
Cryo-ET Data Shape: (64, 64, 64)
Cryo-ET Data Min: 0.0
Cryo-ET Data Max: 0.94504917
Cryo-ET Unique Values: [0.0000000e+00 1.0223870e-20 2.0447740e-20 ... 9.4461203e-01 9.4485116e-01
 9.4504917e-01]
Step 0, Total Loss: 3263.819580, Data Loss: 0.065252, Physics Loss: 244.455231
